# Tech Challenge Fase 2
## Notebook 03 — Gold Orquestrador

### Responsabilidade

Este notebook prepara a execução da camada Gold baseada em **produtos de dados**.

A nova arquitetura Gold será:

```text
03_1_gold_alunos
03_2_gold_municipios
03_3_gold_estados
03_4_gold_indicadores
03_5_gold_powerbi
03_6_gold_machine_learning
```

A base de alunos passa a gerar indicadores agregados por município e UF, enriquecendo as demais visões analíticas.

## 1. Imports

In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

## 2. Leitura do config.json

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

SILVER_PATH = config["paths"]["silver_path"]
GOLD_PATH = config["paths"]["gold_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

## 3. Construção da gold_metadata

In [0]:
gold_metadata = []

def add_metadata(produto, dataset, ano, silver_file_name, gold_output_path, gold_file_name, notebook):
    gold_metadata.append({
        "produto": produto,
        "dataset": dataset,
        "ano": int(ano),
        "silver_path": f"{SILVER_PATH}/{dataset}/ano={ano}" if ano > 0 else "",
        "silver_file_name": silver_file_name,
        "gold_output_path": gold_output_path,
        "gold_file_name": gold_file_name,
        "notebook": notebook
    })

for ano in [2023, 2024, 2025]:
    add_metadata(
        "gold_alunos",
        "alunos",
        ano,
        f"TS_ALUNO_{ano}.csv",
        f"{GOLD_PATH}/alunos/ano={ano}",
        f"GOLD_ALUNOS_{ano}.csv",
        "03_1_gold_alunos"
    )

    add_metadata(
        "gold_municipios",
        "municipios",
        ano,
        f"TS_MUNICIPIO_{ano}.csv",
        f"{GOLD_PATH}/municipios/ano={ano}",
        f"GOLD_MUNICIPIOS_{ano}.csv",
        "03_2_gold_municipios"
    )

    add_metadata(
        "gold_municipios",
        "metas_municipios",
        ano,
        f"TS_METAS_MUNICIPIOS_{ano}_SILVER.csv",
        f"{GOLD_PATH}/municipios/ano={ano}",
        f"GOLD_MUNICIPIOS_{ano}.csv",
        "03_2_gold_municipios"
    )

    add_metadata(
        "gold_estados",
        "estados",
        ano,
        f"TS_ESTADO_{ano}_SILVER.csv",
        f"{GOLD_PATH}/estados/ano={ano}",
        f"GOLD_ESTADOS_{ano}.csv",
        "03_3_gold_estados"
    )

    add_metadata(
        "gold_estados",
        "metas_ufs",
        ano,
        f"TS_METAS_UFS_{ano}_SILVER.csv",
        f"{GOLD_PATH}/estados/ano={ano}",
        f"GOLD_ESTADOS_{ano}.csv",
        "03_3_gold_estados"
    )

gold_metadata.extend([
    {
        "produto":"gold_indicadores",
        "dataset":"gold_alunos",
        "ano":0,
        "silver_path":"",
        "silver_file_name":"",
        "gold_output_path":f"{GOLD_PATH}/indicadores",
        "gold_file_name":"GOLD_INDICADORES.csv",
        "notebook":"03_4_gold_indicadores"
    },
    {
        "produto":"gold_powerbi",
        "dataset":"gold_indicadores",
        "ano":0,
        "silver_path":"",
        "silver_file_name":"",
        "gold_output_path":f"{GOLD_PATH}/exports_powerbi",
        "gold_file_name":"POWERBI_BASE_CONSOLIDADA.csv",
        "notebook":"03_5_gold_powerbi"
    },
    {
        "produto":"gold_machine_learning",
        "dataset":"gold_indicadores",
        "ano":0,
        "silver_path":"",
        "silver_file_name":"",
        "gold_output_path":f"{GOLD_PATH}/base_modelo_ia",
        "gold_file_name":"BASE_MODELO_IA.csv",
        "notebook":"03_6_gold_machine_learning"
    }
])

## Regras de granularidade dos produtos Gold

Os produtos devem respeitar as seguintes chaves únicas:

```text
Gold Alunos:      ANO + CO_MUNICIPIO
Gold Municípios:  ANO + CO_MUNICIPIO
Gold Estados:     ANO + CO_UF
```

Os notebooks filhos validam essas granularidades antes da persistência. A execução deve ser interrompida caso um join produza relação muitos-para-muitos.

## 4. Persistência da gold_metadata

In [0]:
schema = StructType([
    StructField("produto", StringType(), True),
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("silver_path", StringType(), True),
    StructField("silver_file_name", StringType(), True),
    StructField("gold_output_path", StringType(), True),
    StructField("gold_file_name", StringType(), True),
    StructField("notebook", StringType(), True)
])

df_gold_metadata = spark.createDataFrame(gold_metadata, schema=schema)

metadata_path = f"{CONFIG_PATH}/gold_metadata"

(
    df_gold_metadata
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(metadata_path)
)

display(df_gold_metadata.orderBy("produto", "dataset", "ano"))

## 5. Validação da Silver

In [0]:
validation_results = []

for item in gold_metadata:
    if item["ano"] == 0:
        continue

    expected_file = f"{item['silver_path']}/{item['silver_file_name']}"

    try:
        dbutils.fs.ls(expected_file)
        status = "OK"
        error_message = ""
    except Exception as e:
        status = "PENDENTE"
        error_message = str(e)

    validation_results.append({
        "produto": str(item["produto"]),
        "dataset": str(item["dataset"]),
        "ano": int(item["ano"]),
        "expected_file": str(expected_file),
        "status": str(status),
        "error_message": str(error_message),
        "execution_date": str(EXECUTION_DATE)
    })

schema_validation = StructType([
    StructField("produto", StringType(), True),
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("expected_file", StringType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("execution_date", StringType(), True)
])

df_validation = spark.createDataFrame(validation_results, schema=schema_validation)

display(df_validation.orderBy("produto", "dataset", "ano"))

## 6. Checklist final

In [0]:
pendentes = df_validation.filter(F.col("status") == "PENDENTE").count()

if pendentes > 0:
    display(df_validation.filter(F.col("status") == "PENDENTE"))
    raise Exception(f"Existem {pendentes} arquivos Silver pendentes.")

print("Gold Orquestrador concluído com sucesso.")